# Exercise 3:
<b>Instructions</b>
* Keep a subset of columns and clean up column names (no spaces, newlines, etc):
    * columns related to identifying the agency
    * population, passenger trips
    * transit mode
    * at least 3 service metric variables, normalized and not normalized
* Deal with duplicates - what is the unit for each row? What is the unit for desired analysis? Should an agency appear multiple times, and if so, why?
* Aggregate at least 2 ways and show an interesting comparison, after dealing with duplicates somehow (either aggregation and/or defining what the unit of analysis is)
* Calculate weighted average after the aggregation for the service metrics
* Decide on one type of chart to visualize, and generalize it as a function
* Make charts using the function

### Helpful Hints for Functions
* Opportunities are from components that are generalizable in making a chart
* Maybe these components need the same lines of code to clean them
* You can always further define variables within a function
* You can always use f-strings within functions to make slight modifications to the parameters you pass

## Reading in my file & cleaning it up

In [1]:
import pandas as pd
import numpy as np
import shared_utils
import altair as alt
from shared_utils import altair_utils 

pd.set_option('display.max_columns', None)

/opt/conda/lib/python3.9/site-packages/geopandas/_compat.py:111: UserWarning: The Shapely GEOS version (3.9.1-CAPI-1.14.2) is incompatible with the GEOS version PyGEOS was compiled with (3.10.1-CAPI-1.16.0). Conversions between both will be slow.
  warnings.warn(


In [2]:
GCS_FILE_PATH = "gs://calitp-analytics-data/data-analyses/bus_service_increase/"
FILE_NAME = "ntd_metrics_2019.csv"
df2 = pd.read_csv(f"{GCS_FILE_PATH}{FILE_NAME}")

In [3]:
#cleaning up columns
df2.columns = df2.columns.str.strip().str.replace(' ', '_')

## Subsetting for the 3 performance metrics & agency identifiers.

<b> Just notes on the different abbreviations
* TOS: Types of Service

* MB: Driving a bus

* DO: Vehicle Maintenance 

* VOMS: vehicles operated in annual maximum service

In [4]:
list(df2.columns)

['Agency',
 'City',
 'State',
 'Legacy_NTD_ID',
 'NTD_ID',
 'Organization_Type',
 'Reporter_Type',
 'Primary_UZA\n_Population',
 'Agency_VOMS',
 'Mode',
 'TOS',
 'Mode_VOMS',
 'Ratios:',
 'Fare_Revenues_per_Unlinked_Passenger_Trip',
 'Fare_Revenues_per_Unlinked_Passenger_Trip_Questionable',
 'Fare_Revenues_per_Total_Operating_Expense_(Recovery_Ratio)',
 'Fare_Revenues_per_Total_Operating_Expense_(Recovery_Ratio)_Questionable',
 'Cost_per\n_Hour',
 'Cost_per_Hour_Questionable',
 'Passengers_per_Hour',
 'Passengers_per_Hour_Questionable',
 'Cost_per_Passenger',
 'Cost_per_Passenger_Questionable',
 'Cost_per_Passenger_Mile',
 'Cost_per_Passenger_Mile_Questionable',
 'Source_Data:',
 'Fare_Revenues_Earned',
 'Fare_Revenues_Earned_Questionable',
 'Total_Operating_Expenses',
 'Total_Operating_Expenses_Questionable',
 'Unlinked_Passenger_Trips',
 'Unlinked_Passenger_Trips_Questionable',
 'Vehicle_Revenue_Hours',
 'Vehicle_Revenue_Hours_Questionable',
 'Passenger_Miles',
 'Passenger_Miles_Ques

In [5]:
df3 = df2[['Agency', 'City', 'State', 'Legacy_NTD_ID', 'NTD_ID',
    'Primary_UZA\n_Population', 'Mode','Cost_per\n_Hour','Passengers_per_Hour','Cost_per_Passenger','TOS','Unlinked_Passenger_Trips','Total_Operating_Expenses']]

In [6]:
df3 = df3.rename(columns = {'Primary_UZA\n_Population': 'Primary_Population', 'Cost_per\n_Hour': 'Cost_per_hour'}) #renaming columns 

In [7]:
print(f"There are: {len(df3)} rows before any filtering")

There are: 3685 rows before any filtering


In [8]:
df3.head(2)

,Agency,City,State,Legacy_NTD_ID,NTD_ID,Primary_Population,Mode,Cost_per_hour,Passengers_per_Hour,Cost_per_Passenger,TOS,Unlinked_Passenger_Trips,Total_Operating_Expenses
0,MTA New York City Transit,New York,NY,2008,20008,"18,351,295",HR,$267.97,139.6,$1.92,DO,"2,712,521,697","$5,206,727,193"
1,MTA New York City Transit,New York,NY,2008,20008,"18,351,295",CB,$393.55,18.6,$21.13,DO,"11,477,164","$242,520,835"


## Deal with Duplicates: What is the unit for each row? What is the unit for desired analysis? Should an agency appear multiple times, and if so, why?

* The unit for each row is the mode of transportation by agency & geography, looking at the cost per hour of operating the vehicle, how many passengers boarded per hour, total unlinked trips, cost per passenger, etc. 

* An agency should appear multiple times as some modes of transportation are contracted out to either vendors. Additionally, there are agencies in different states that have similar or even the same names.

In [9]:
#looking at duplicates...these are just NA values. 
duplicateRowsDF = df3[df3.duplicated()]
duplicateRowsDF

,Agency,City,State,Legacy_NTD_ID,NTD_ID,Primary_Population,Mode,Cost_per_hour,Passengers_per_Hour,Cost_per_Passenger,TOS,Unlinked_Passenger_Trips,Total_Operating_Expenses
3681,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3682,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3683,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3684,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
#dropping NA
df3 = df3.dropna()

In [11]:
print(f"Number of unique agencies after we deduplicated: {df3.Agency.nunique()}")

Number of unique agencies after we deduplicated: 1955


In [12]:
print(f"There are: {len(df3)} rows after  dropping duplicates")

There are: 3394 rows after  dropping duplicates


In [13]:
#looking at the few of the agencies that still have duplicates to see why...
df3[(df3.Agency.str.contains("Southern Teton Area Rapid Transit"))]

,Agency,City,State,Legacy_NTD_ID,NTD_ID,Primary_Population,Mode,Cost_per_hour,Passengers_per_Hour,Cost_per_Passenger,TOS,Unlinked_Passenger_Trips,Total_Operating_Expenses
1759,Southern Teton Area Rapid Transit,Jackson,WY,8R05-010,8R05-80188,0,DR,$22.40,1.0,$21.46,DO,"5,244","$112,528"
1761,Southern Teton Area Rapid Transit,Jackson,WY,8R05-010,8R05-80188,0,MB,$71.66,20.3,$3.53,DO,"1,098,224","$3,875,238"
3384,Southern Teton Area Rapid Transit,Jackson,WY,8R05-010,0R01-80188,0,MB,$146.39,15.4,$9.52,DO,"33,383","$317,816"


In [14]:
#looking at the few of the agencies that still have duplicates to see why...some agencies have the same name in didfferent states
df3[(df3.Agency.str.contains("Union County Transit"))]

,Agency,City,State,Legacy_NTD_ID,NTD_ID,Primary_Population,Mode,Cost_per_hour,Passengers_per_Hour,Cost_per_Passenger,TOS,Unlinked_Passenger_Trips,Total_Operating_Expenses
2849,Union County Transit,LIBERTY,IN,5R02-038,5R02-50387,0,DR,$34.50,1.7,$20.13,DO,"24,582","$494,890"
3550,Union County Transit,Blairsville,GA,4R03-012,4R03-41145,0,DR,$22.69,1.6,$14.24,DO,"6,180","$87,989"


In [15]:
#some agencies have VERY similar names in different states...The transit agency Jackson County appears in both NC and GA
df3[(df3.Agency.str.contains("Jackson County"))]

,Agency,City,State,Legacy_NTD_ID,NTD_ID,Primary_Population,Mode,Cost_per_hour,Passengers_per_Hour,Cost_per_Passenger,TOS,Unlinked_Passenger_Trips,Total_Operating_Expenses
1526,"Jackson County Transportation, Inc.",Marianna,FL,4R02-019,4R02-41198,0,DR,$52.98,1.4,$38.23,DO,"37,567","$1,436,173"
1557,"Jackson County Transportation, Inc.",Marianna,FL,4R02-019,4R02-41198,0,MB,$31.76,2.8,$11.49,DO,"2,195","$25,216"
1981,"Delaware, Dubuque & Jackson County Regional Tr...",Dubuque,IA,7R01-008,7R01-70136,0,DR,$66.81,3.6,$18.55,DO,"99,025","$1,837,191"
2047,Jackson County Mass Transit District,Carbondale,IL,5204,50204,"67,821",MB,$47.76,3.3,$14.67,DO,"93,691","$1,374,554"
2614,Jackson County,Sylva,NC,4R06-023,4R06-41167,0,DR,$47.40,1.6,$28.92,DO,"18,663","$539,672"
2660,Jackson County,Sylva,NC,4R06-023,4R06-41167,0,MB,$47.58,3.0,$15.69,DO,"8,004","$125,576"
2953,Jackson County Council on Aging,Scottsboro,AL,4R01-016,4R01-41180,0,DR,$40.65,2.9,$14.25,DO,"29,346","$418,250"
3370,Jackson County,Jefferson,GA,4R03-009,4R03-41154,0,DR,$23.39,1.7,$14.09,DO,"12,285","$173,101"


In [16]:
#same thing, there's a Valley Transit in Arkansas and in Washington.
df3[(df3.Agency.str.contains("Valley Transit"))]

,Agency,City,State,Legacy_NTD_ID,NTD_ID,Primary_Population,Mode,Cost_per_hour,Passengers_per_Hour,Cost_per_Passenger,TOS,Unlinked_Passenger_Trips,Total_Operating_Expenses
328,Victor Valley Transit Authority,Hesperia,CA,9148,90148,"328,454",CB,$109.56,5.0,$22.06,PT,"35,756","$788,860"
329,Victor Valley Transit Authority,Hesperia,CA,9148,90148,"328,454",MB,$85.55,7.5,$11.39,PT,"1,442,723","$16,436,694"
330,Victor Valley Transit Authority,Hesperia,CA,9148,90148,"328,454",DR,$89.98,3.1,$29.38,PT,"189,182","$5,557,594"
331,Victor Valley Transit Authority,Hesperia,CA,9148,90148,"328,454",VP,$32.93,5.3,$6.21,PT,"572,713","$3,555,942"
387,Pioneer Valley Transit Authority,Springfield,MA,1008,10008,"621,300",MB,$106.10,27.4,$3.88,PT,"10,120,344","$39,216,072"
388,Pioneer Valley Transit Authority,Springfield,MA,1008,10008,"621,300",DR,$53.30,1.4,$38.27,PT,"260,582","$9,973,167"
514,Minnesota Valley Transit Authority,Burnsville,MN,5222,50519,"2,650,890",MB,$158.92,15.0,$10.62,PT,"2,547,655","$27,060,351"
684,Antelope Valley Transit Authority,Lancaster,CA,9121,90121,"341,219",MB,$121.34,13.0,$9.32,PT,"2,013,685","$18,771,318"
685,Antelope Valley Transit Authority,Lancaster,CA,9121,90121,"341,219",CB,$134.13,9.3,$14.37,PT,"288,183","$4,142,476"
686,Antelope Valley Transit Authority,Lancaster,CA,9121,90121,"341,219",DR,$56.05,2.3,$24.88,PT,"50,600","$1,258,678"


## Changing mode abbrevations 
* Mapping a few of the abbreviations to the full name for ease.

* [Transit.dot.gov's data dictionary](https://www.transit.dot.gov/ntd/national-transit-database-ntd-glossary#C)

#### Trying to map abbreviations with df.apply
Write df.apply statement lambda function.

Tiffany's code

```
df["mode_full_name"] = df.apply(lambda x: "Other" if x.mode_full_name=="" else x.mode_full_name, axis=1) #axis=1, is row

RAIL = ["LR", "HR", "CR"]
BUS = ["CB", "MB", "AR", "DR"]

def replace_modes(row):
    if RAIL in row.Mode:
        return "rail"
    elif BUS in row.Mode:
        return "bus"
    else:
        return "other"
        
df["mode_full_name"] = df.apply(lambda x: replace_modes(x), axis=1)
```

In [17]:
df3.Mode.unique().tolist()

['HR',
 'CB',
 'MB',
 'DR',
 'RB',
 'CR',
 'LR',
 'VP',
 'YR',
 'DT',
 'FB',
 'TB',
 'SR',
 'PB',
 'MG',
 'CC',
 'IP',
 'TR',
 'AR']

In [18]:
#creating a list for all rail and all bus related modes of transportation.
RAIL = ['HR',
'DR', 'CR', 'LR',
'YR', 'SR',
'TR',
'AR']

BUS = [
 'CB',
 'MB',
 'RB',
 'FB',
 'TB','PB']

### HELP This didn't work for me, error message says "TypeError: 'in <string>' requires string as left operand, not list" but when I convert it into a string, it just puts everything in the other category.

In [19]:
def replace_modes(row):
    if row.Mode in RAIL:
        return "Rail"
    elif row.Mode in BUS:
        return "Bus"
    else:
        return "Other"
        

In [20]:
df3["mode_full_name_test"] = df3.apply(lambda x: replace_modes(x), axis=1) 


In [21]:
df3.head(3)

,Agency,City,State,Legacy_NTD_ID,NTD_ID,Primary_Population,Mode,Cost_per_hour,Passengers_per_Hour,Cost_per_Passenger,TOS,Unlinked_Passenger_Trips,Total_Operating_Expenses,mode_full_name_test
0,MTA New York City Transit,New York,NY,2008,20008,"18,351,295",HR,$267.97,139.6,$1.92,DO,"2,712,521,697","$5,206,727,193",Rail
1,MTA New York City Transit,New York,NY,2008,20008,"18,351,295",CB,$393.55,18.6,$21.13,DO,"11,477,164","$242,520,835",Bus
2,MTA New York City Transit,New York,NY,2008,20008,"18,351,295",MB,$219.87,56.6,$3.88,DO,"691,616,614","$2,685,918,268",Bus


#### Trying to map it neater 

In [22]:
df3['Full_Mode_Name'] = np.nan

In [23]:
df3.loc[df3.Mode.isin(['CB','MB','RB','FB','TB','PB']),  'Full_Mode_Name'] = 'Bus'
df3.loc[df3.Mode.isin(['HR','DR','CR','LR','YR','SR','TR','AR']), 'Full_Mode_Name'] = 'Rail'

In [24]:
df3[['Full_Mode_Name']] = df3[['Full_Mode_Name']].fillna(value= 'Other')

In [25]:
df3.Full_Mode_Name.value_counts()

Rail     1843
Bus      1346
Other     205
Name: Full_Mode_Name, dtype: int64

## Cleaning up columns

In [26]:
(df3.columns)

Index(['Agency', 'City', 'State', 'Legacy_NTD_ID', 'NTD_ID',
       'Primary_Population', 'Mode', 'Cost_per_hour', 'Passengers_per_Hour',
       'Cost_per_Passenger', 'TOS', 'Unlinked_Passenger_Trips',
       'Total_Operating_Expenses', 'mode_full_name_test', 'Full_Mode_Name'],
      dtype='object')

In [27]:
#cleaning up string columns
for i in ['Agency', 'City', 'State', 'Full_Mode_Name']:
    df3[i] = df3[i].str.strip().replace(',', '').astype({i: str})

In [28]:
df3["Cost_per_Passenger"].replace({"1,218.00": "1218"}, inplace=True)

In [29]:
#cleaning up the numeric columns
for i in ['Primary_Population', 'Cost_per_hour', 'Passengers_per_Hour',
       'Cost_per_Passenger', 'Unlinked_Passenger_Trips', 'Total_Operating_Expenses']:
    df3[i] = df3[i].replace({'\$':''}, regex=True).replace(',','', regex= True).astype({i: float})

In [30]:
#making sure all the data types are accurate
df3.dtypes

Agency                       object
City                         object
State                        object
Legacy_NTD_ID                object
NTD_ID                       object
Primary_Population          float64
Mode                         object
Cost_per_hour               float64
Passengers_per_Hour         float64
Cost_per_Passenger          float64
TOS                          object
Unlinked_Passenger_Trips    float64
Total_Operating_Expenses    float64
mode_full_name_test          object
Full_Mode_Name               object
dtype: object

In [31]:
#dropping the old mode 
df3 = df3.drop(columns = ['Mode', 'mode_full_name_test'])

In [32]:
df3.to_csv("./df3.csv", index= False) #just exporting to CSV so I can check it out

# Aggregation & Chart Making

### First Analysis


In [33]:
#filter out just bus for mode
bus = df3[df3["Full_Mode_Name"] == "Bus"]

In [34]:
bus.head(2)

,Agency,City,State,Legacy_NTD_ID,NTD_ID,Primary_Population,Cost_per_hour,Passengers_per_Hour,Cost_per_Passenger,TOS,Unlinked_Passenger_Trips,Total_Operating_Expenses,Full_Mode_Name
1,MTA New York City Transit,New York,NY,2008,20008,18351295.0,393.55,18.6,21.13,DO,11477164.0,2.425208e+08,Bus
2,MTA New York City Transit,New York,NY,2008,20008,18351295.0,219.87,56.6,3.88,DO,691616614.0,2.685918e+09,Bus


In [35]:
first= bus.drop_duplicates().groupby(['State', 'Full_Mode_Name']).agg({'Unlinked_Passenger_Trips': 'sum', 'Total_Operating_Expenses': 'sum' }).reset_index()
first.sort_values(by="Unlinked_Passenger_Trips", ascending= False)

,State,Full_Mode_Name,Unlinked_Passenger_Trips,Total_Operating_Expenses
36,NY,Bus,1.026697e+09,4.810735e+09
5,CA,Bus,8.455713e+08,4.665400e+09
16,IL,Bus,2.937754e+08,1.210445e+09
40,PA,Bus,2.414856e+08,1.187525e+09
51,WA,Bus,2.234103e+08,1.613615e+09
46,TX,Bus,2.147267e+08,1.310105e+09
33,NJ,Bus,1.883524e+08,1.220660e+09
10,FL,Bus,1.795735e+08,1.076548e+09
21,MA,Bus,1.431641e+08,6.505592e+08
8,DC,Bus,1.287985e+08,7.587671e+08


In [36]:
first = first.nlargest(10,'Unlinked_Passenger_Trips')
first["Operating_Expenses_to_Trips"] = first.Total_Operating_Expenses.divide(first.Unlinked_Passenger_Trips)
 
first

,State,Full_Mode_Name,Unlinked_Passenger_Trips,Total_Operating_Expenses,Operating_Expenses_to_Trips
36,NY,Bus,1.026697e+09,4.810735e+09,4.685643
5,CA,Bus,8.455713e+08,4.665400e+09,5.517453
16,IL,Bus,2.937754e+08,1.210445e+09,4.120307
40,PA,Bus,2.414856e+08,1.187525e+09,4.917579
51,WA,Bus,2.234103e+08,1.613615e+09,7.222654
46,TX,Bus,2.147267e+08,1.310105e+09,6.101268
33,NJ,Bus,1.883524e+08,1.220660e+09,6.480721
10,FL,Bus,1.795735e+08,1.076548e+09,5.995027
21,MA,Bus,1.431641e+08,6.505592e+08,4.544150
8,DC,Bus,1.287985e+08,7.587671e+08,5.891117


In [37]:
state_dropdown = first.State.unique().tolist()


In [38]:
input_dropdown = alt.binding_select(options=state_dropdown)
selection = alt.selection_single(fields=['State'], bind=input_dropdown, name='State')

In [39]:
#Just saving this because of the drop down menu 
#def bar_chart(df, x_col, y_col):
   # chart = (alt.Chart(df).mark_bar().encode(
           #      x=alt.X(x_col, title=x_col),
            #     y=alt.Y(y_col, title=y_col),   
              #   color =alt.Color(x_col, scale=alt.Scale(range=altair_utils.FIVETHIRTYEIGHT_CATEGORY_COLORS)),
                # tooltip = [alt.Tooltip(x_col),alt.Tooltip(y_col)])#.add_selection(selection).transform_filter(selection)
          #  )
    
  #  return chart

### HELP I feel like the colors are not working?

In [40]:
def bar_chart(df, x_col, y_col):
    chart = (alt.Chart(df).mark_bar().encode(
                 x=alt.X(x_col, title=x_col),
                 y=alt.Y(y_col, title=y_col),   
                 color =alt.Color(x_col, scale=alt.Scale(range=altair_utils.FIVETHIRTYEIGHT_CATEGORY_COLORS)),
                 tooltip = [alt.Tooltip(x_col),alt.Tooltip(y_col)]).interactive().properties(width=600,height=250)
            )
    
    return chart

In [41]:
chart_one = bar_chart(first, 'State', 'Operating_Expenses_to_Trips')
chart_one

alt.Chart(...)

### Second Analysis

In [42]:
 second = df3.groupby(['Full_Mode_Name']).agg({'Unlinked_Passenger_Trips': 'median', 'Total_Operating_Expenses': 'median' }).reset_index()

In [43]:
second.head(2)

,Full_Mode_Name,Unlinked_Passenger_Trips,Total_Operating_Expenses
0,Bus,196085.0,1671697.5
1,Other,66002.0,513227.0


In [44]:
chart_two = bar_chart(second, 'Full_Mode_Name', 'Total_Operating_Expenses')
chart_two 

alt.Chart(...)

In [45]:
chart_three = bar_chart(second, 'Full_Mode_Name', 'Unlinked_Passenger_Trips')
chart_three

alt.Chart(...)